# Consolidated Modeling: 

## Loading Libraries and Preprocessing:

In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.model_selection import (
    BaseCrossValidator,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# Load your dataset
df = pd.read_csv("financial_regression.csv")

# Convert date
df["date"] = pd.to_datetime(df["date"])

# Sort chronologically
df = df.sort_values("date").reset_index(drop=True)

# Remove days where gold was not traded
df = df.dropna(subset=["gold close"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

Dataset shape: (3719, 47)
Date range: 2010-01-14 00:00:00 to 2024-10-23 00:00:00


In [3]:
# ---------------------------------------------------------
# Basic lagged features
# ---------------------------------------------------------

df["Close_Lag1"] = df["gold close"].shift(1)
df["Open_Lag1"] = df["gold open"].shift(1)

# Today's open-to-close return
df["Daily_Return"] = (
    (df["gold close"] - df["gold open"])
    / df["gold open"]
)

# ---------------------------------------------------------
# Calendar features
# ---------------------------------------------------------

df["DayOfWeek"] = df["date"].dt.dayofweek
df["Month"] = df["date"].dt.month
df["Quarter"] = df["date"].dt.quarter
df["DayOfYear"] = df["date"].dt.dayofyear
df["IsMonthEnd"] = df["date"].dt.is_month_end.astype(int)

print("Basic features created.")

Basic features created.


In [4]:
# Tomorrow's gold closing price
df["Target_Next_Close"] = df["gold close"].shift(-1)

In [5]:
# =========================================================
# ADVANCED LOG-RETURN FEATURES
# =========================================================

target_assets = [
    "gold close",
    "silver close",
    "oil close",
    "sp500 close",
    "nasdaq close"
]

lags = [1, 2, 5, 10]

# ---------------------------------------------------------
# Multi-day log returns
# ---------------------------------------------------------

for asset in target_assets:

    col_name = asset.replace(" ", "_")

    for lag in lags:

        df[f"{col_name}_log_ret_{lag}d"] = np.log(
            df[asset] / df[asset].shift(lag)
        )


# ---------------------------------------------------------
# Gold volatility
# ---------------------------------------------------------

gold_ret = df["gold_close_log_ret_1d"]

df["gold_volatility_5d"] = (
    gold_ret.rolling(window=5).std()
)

df["gold_volatility_20d"] = (
    gold_ret.rolling(window=20).std()
)


# ---------------------------------------------------------
# Gold distance from 20-day moving average
# ---------------------------------------------------------

gold_sma20 = (
    df["gold close"]
    .rolling(window=20)
    .mean()
)

df["gold_sma20_dist"] = np.log(
    df["gold close"] / gold_sma20
)


# ---------------------------------------------------------
# Gold/Silver ratio
# ---------------------------------------------------------

df["gold_silver_ratio"] = (
    df["gold close"] / df["silver close"]
)

df["gold_silver_ratio_chg"] = np.log(
    df["gold_silver_ratio"]
    / df["gold_silver_ratio"].shift(1)
)


# ---------------------------------------------------------
# Gold/Oil ratio
# ---------------------------------------------------------

df["gold_oil_ratio"] = (
    df["gold close"] / df["oil close"]
)

df["gold_oil_ratio_chg"] = np.log(
    df["gold_oil_ratio"]
    / df["gold_oil_ratio"].shift(1)
)


# ---------------------------------------------------------
# Gold volume change
# ---------------------------------------------------------

df["gold_volume_change"] = np.log(
    (df["gold volume"] + 1)
    / (df["gold volume"].shift(1) + 1)
)

In [6]:
# ---------------------------------------------------------
# Calendar / cyclical features
# ---------------------------------------------------------

df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

df["DayOfYear_sin"] = np.sin(
    2 * np.pi * df["DayOfYear"] / 365.25
)

df["DayOfYear_cos"] = np.cos(
    2 * np.pi * df["DayOfYear"] / 365.25
)

In [7]:
# =========================================================
# NEXT-DAY LOG RETURN TARGET
# =========================================================

df["Target_Log_Return"] = np.log(
    df["gold close"].shift(-1)
    / df["gold close"]
)

In [ ]:
# =========================================================
# DIRECTIONAL TARGET
# =========================================================

df["Target_Direction"] = (
    df["Target_Log_Return"] > 0
).astype(int)

print(
    df["Target_Direction"]
    .value_counts(normalize=True)
)

#1 = tomorrow's return is positive
#0 = tomorrow's return is zero or negative

Target_Direction
1    0.522452
0    0.477548
Name: proportion, dtype: float64


In [9]:
df = df.dropna(
    subset=[
        "Target_Next_Close",
        "Target_Log_Return"
    ]
).reset_index(drop=True)

print("Final dataset shape:", df.shape)

Final dataset shape: (3718, 90)


In [10]:
raw_numeric_features = [
    "sp500 open",
    "sp500 high",
    "sp500 low",
    "sp500 close",

    "nasdaq open",
    "nasdaq high",
    "nasdaq low",
    "nasdaq close",

    "silver close",
    "oil close",
    "platinum close",
    "palladium close",

    "gold open",
    "gold high",
    "gold low",
    "gold close",
    "gold volume",

    "Close_Lag1",
    "Open_Lag1",
    "Daily_Return"
]

raw_categorical_features = [
    "DayOfWeek",
    "Month",
    "Quarter",
    "DayOfYear",
    "IsMonthEnd"
]

raw_features = (
    raw_numeric_features
    + raw_categorical_features
)

raw_target = "Target_Next_Close"

In [11]:
log_numeric_features = [
    "Daily_Return",

    # 1-day returns
    "gold_close_log_ret_1d",
    "silver_close_log_ret_1d",
    "oil_close_log_ret_1d",
    "sp500_close_log_ret_1d",
    "nasdaq_close_log_ret_1d",

    # 2-day returns
    "gold_close_log_ret_2d",
    "silver_close_log_ret_2d",
    "oil_close_log_ret_2d",
    "sp500_close_log_ret_2d",
    "nasdaq_close_log_ret_2d",

    # 5-day returns
    "gold_close_log_ret_5d",
    "silver_close_log_ret_5d",
    "oil_close_log_ret_5d",
    "sp500_close_log_ret_5d",
    "nasdaq_close_log_ret_5d",

    # 10-day returns
    "gold_close_log_ret_10d",
    "silver_close_log_ret_10d",
    "oil_close_log_ret_10d",
    "sp500_close_log_ret_10d",
    "nasdaq_close_log_ret_10d",

    # Volatility
    "gold_volatility_5d",
    "gold_volatility_20d",

    # Trend
    "gold_sma20_dist",

    # Cross-market
    "gold_silver_ratio_chg",
    "gold_oil_ratio_chg",

    # Volume
    "gold_volume_change",

    # Cyclical time
    "Month_sin",
    "Month_cos",
    "DayOfYear_sin",
    "DayOfYear_cos"
]

log_categorical_features = [
    "DayOfWeek",
    "IsMonthEnd"
]

log_features = (
    log_numeric_features
    + log_categorical_features
)

log_target = "Target_Log_Return"
classification_target = "Target_Direction"

In [13]:
class PurgedTimeSeriesCV(BaseCrossValidator):
    """
    Expanding-window time-series CV with a purge/embargo gap.

    Training observations always occur before validation observations.

    Parameters
    ----------
    n_splits : int
        Number of validation folds.

    test_size : int
        Number of observations in each validation fold.

    purge_size : int
        Number of observations removed immediately before
        the validation set.

    embargo_size : int
        Additional gap between training and validation.
        For a strictly chronological walk-forward split,
        this acts as an additional gap before validation.
    """

    def __init__(
        self,
        n_splits=3,
        test_size=200,
        purge_size=1,
        embargo_size=1
    ):
        self.n_splits = n_splits
        self.test_size = test_size
        self.purge_size = purge_size
        self.embargo_size = embargo_size

    def split(self, X, y=None, groups=None):

        n_samples = len(X)

        # Total gap between train and validation
        gap = (
            self.purge_size
            + self.embargo_size
        )

        # Last possible validation fold
        first_test_start = (
            n_samples
            - self.n_splits * self.test_size
        )

        if first_test_start <= gap:
            raise ValueError(
                "Not enough observations for the "
                "requested number of splits."
            )

        for i in range(self.n_splits):

            test_start = (
                first_test_start
                + i * self.test_size
            )

            test_end = (
                test_start
                + self.test_size
            )

            train_end = test_start - gap

            train_indices = np.arange(
                0,
                train_end
            )

            test_indices = np.arange(
                test_start,
                min(test_end, n_samples)
            )

            yield train_indices, test_indices

    def get_n_splits(
        self,
        X=None,
        y=None,
        groups=None
    ):
        return self.n_splits

In [ ]:
# 2010–2013 → 2014
# 2011–2014 → 2015
# 2012–2015 → 2016
# 2013–2016 → 2017
# 2014–2017 → 2018
# 2015–2018 → 2019
# 2016–2019 → 2020
# 2017–2020 → 2021
# 2018–2021 → 2022
# 2019–2022 → 2023
# 2020–2023 → 2024*

In [14]:
def generate_purged_walk_forward_windows(
    df,
    train_years=4,
    test_start_year=2014,
    test_end_year=2024,
    purge_size=1,
    embargo_size=1
):
    """
    Generate chronological walk-forward windows.

    Each fold uses:
        4 years training
        purge + embargo gap
        1 calendar year testing

    The final test year (2024) may be a partial year
    because the dataset ends on 2024-10-23.
    """

    windows = []

    for test_year in range(
        test_start_year,
        test_end_year + 1
    ):

        train_start = test_year - train_years
        train_end = test_year - 1

        train_start_date = pd.Timestamp(
            f"{train_start}-01-01"
        )

        train_end_date = pd.Timestamp(
            f"{train_end}-12-31"
        )

        test_start_date = pd.Timestamp(
            f"{test_year}-01-01"
        )

        test_end_date = pd.Timestamp(
            f"{test_year}-12-31"
        )

        train_mask = (
            (df["date"] >= train_start_date)
            &
            (df["date"] <= train_end_date)
        )

        test_mask = (
            (df["date"] >= test_start_date)
            &
            (df["date"] <= test_end_date)
        )

        train_indices = np.where(train_mask)[0]
        test_indices = np.where(test_mask)[0]

        if len(train_indices) == 0 or len(test_indices) == 0:
            continue

        # Total observations removed between
        # training and testing
        gap = purge_size + embargo_size

        if len(train_indices) <= gap:
            continue

        train_indices = train_indices[:-gap]

        windows.append({
            "train_start": train_start,
            "train_end": train_end,
            "test_year": test_year,
            "train_indices": train_indices,
            "test_indices": test_indices
        })

    return windows

In [15]:
outer_windows = generate_purged_walk_forward_windows(
    df,
    train_years=4,
    test_start_year=2014,
    test_end_year=2024,
    purge_size=1,
    embargo_size=1
)

print(
    f"Generated {len(outer_windows)} "
    "purged walk-forward windows."
)

for w in outer_windows:

    train = df.iloc[w["train_indices"]]
    test = df.iloc[w["test_indices"]]

    print(
        f"Train: {train['date'].min().date()} "
        f"to {train['date'].max().date()} | "
        f"Test: {test['date'].min().date()} "
        f"to {test['date'].max().date()} | "
        f"Train n={len(train)}, "
        f"Test n={len(test)}"
    )

Generated 11 purged walk-forward windows.
Train: 2010-01-14 to 2013-12-27 | Test: 2014-01-02 to 2014-12-31 | Train n=996, Test n=252
Train: 2011-01-03 to 2014-12-29 | Test: 2015-01-02 to 2015-12-31 | Train n=1004, Test n=252
Train: 2012-01-03 to 2015-12-29 | Test: 2016-01-04 to 2016-12-30 | Train n=1004, Test n=252
Train: 2013-01-02 to 2016-12-28 | Test: 2017-01-03 to 2017-12-29 | Train n=1006, Test n=251
Train: 2014-01-02 to 2017-12-27 | Test: 2018-01-02 to 2018-12-31 | Train n=1005, Test n=251
Train: 2015-01-02 to 2018-12-27 | Test: 2019-01-02 to 2019-12-31 | Train n=1004, Test n=252
Train: 2016-01-04 to 2019-12-27 | Test: 2020-01-02 to 2020-12-31 | Train n=1004, Test n=253
Train: 2017-01-03 to 2020-12-29 | Test: 2021-01-04 to 2021-12-31 | Train n=1005, Test n=252
Train: 2018-01-02 to 2021-12-29 | Test: 2022-01-03 to 2022-12-30 | Train n=1006, Test n=251
Train: 2019-01-02 to 2022-12-28 | Test: 2023-01-03 to 2023-12-29 | Train n=1006, Test n=250
Train: 2020-01-02 to 2023-12-27 | Test:

## Phase 1:

In [16]:
def make_scaled_preprocessor(
    numeric_features,
    categorical_features
):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median"
        )),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent"
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ])

    return ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

In [17]:
def make_tree_preprocessor(
    numeric_features,
    categorical_features
):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median"
        ))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent"
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ])

    return ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

In [18]:
def regression_metrics(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),

        "R²": r2_score(
            y_true,
            y_pred
        )
    }

In [19]:
RIDGE_GRID = {
    "model__alpha": np.logspace(
        -2,
        3,
        10
    )
}

In [20]:
RF_REG_GRID = {
    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        3,
        5,
        8
    ],

    "model__min_samples_leaf": [
        2,
        5,
        10
    ],

    "model__max_features": [
        "sqrt",
        0.5
    ]
}

In [21]:
XGB_REG_GRID = {
    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        2,
        3,
        5
    ],

    "model__learning_rate": [
        0.01,
        0.05
    ],

    "model__subsample": [
        0.7,
        0.9
    ],

    "model__colsample_bytree": [
        0.7,
        0.9
    ],

    "model__reg_lambda": [
        1,
        10
    ]
}

In [22]:
def tune_model(
    pipeline,
    param_grid,
    X_train,
    y_train,
    cv,
    scoring="neg_root_mean_squared_error"
):

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    return search

In [23]:
def build_linear_regression():

    preprocessor = make_scaled_preprocessor(
        raw_numeric_features,
        raw_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ])

In [24]:
def build_ridge_regression():

    preprocessor = make_scaled_preprocessor(
        raw_numeric_features,
        raw_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            Ridge()
        )
    ])

In [25]:
def build_rf_regression():

    preprocessor = make_tree_preprocessor(
        raw_numeric_features,
        raw_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])

In [26]:
def build_xgb_regression():

    preprocessor = make_tree_preprocessor(
        raw_numeric_features,
        raw_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])

In [27]:
raw_results = []

for w in outer_windows:

    train_idx = w["train_indices"]
    test_idx = w["test_indices"]

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    X_train = train[raw_features]
    y_train = train[raw_target]

    X_test = test[raw_features]
    y_test = test[raw_target]

    # -----------------------------------------------------
    # Inner purged CV
    # -----------------------------------------------------

    inner_cv = PurgedTimeSeriesCV(
        n_splits=3,
        test_size=150,
        purge_size=1,
        embargo_size=1
    )

    models = {

        "Linear Regression": (
            build_linear_regression(),
            None
        ),

        "Ridge Regression": (
            build_ridge_regression(),
            RIDGE_GRID
        ),

        "Random Forest": (
            build_rf_regression(),
            RF_REG_GRID
        ),

        "XGBoost": (
            build_xgb_regression(),
            XGB_REG_GRID
        )
    }

    for model_name, (pipeline, grid) in models.items():

        print(
            f"{w['test_year']} | "
            f"{model_name}"
        )

        # -------------------------------------------------
        # Linear Regression has no hyperparameters
        # -------------------------------------------------

        if grid is None:

            pipeline.fit(
                X_train,
                y_train
            )

            best_model = pipeline
            best_params = {}

        else:

            search = tune_model(
                pipeline,
                grid,
                X_train,
                y_train,
                inner_cv
            )

            best_model = search.best_estimator_
            best_params = search.best_params_

        # -------------------------------------------------
        # Test
        # -------------------------------------------------

        y_pred = best_model.predict(
            X_test
        )

        metrics = regression_metrics(
            y_test,
            y_pred
        )

        raw_results.append({
            "Test Year": w["test_year"],
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R²": metrics["R²"],
            "Best Params": best_params
        })

raw_results_df = pd.DataFrame(raw_results)

raw_results_df

2014 | Linear Regression
2014 | Ridge Regression
2014 | Random Forest
2014 | XGBoost
2015 | Linear Regression
2015 | Ridge Regression
2015 | Random Forest
2015 | XGBoost
2016 | Linear Regression
2016 | Ridge Regression
2016 | Random Forest
2016 | XGBoost
2017 | Linear Regression
2017 | Ridge Regression
2017 | Random Forest
2017 | XGBoost
2018 | Linear Regression
2018 | Ridge Regression
2018 | Random Forest
2018 | XGBoost
2019 | Linear Regression
2019 | Ridge Regression
2019 | Random Forest
2019 | XGBoost
2020 | Linear Regression
2020 | Ridge Regression
2020 | Random Forest
2020 | XGBoost
2021 | Linear Regression
2021 | Ridge Regression
2021 | Random Forest
2021 | XGBoost
2022 | Linear Regression
2022 | Ridge Regression
2022 | Random Forest
2022 | XGBoost
2023 | Linear Regression
2023 | Ridge Regression
2023 | Random Forest
2023 | XGBoost
2024 | Linear Regression
2024 | Ridge Regression
2024 | Random Forest
2024 | XGBoost


,Test Year,Model,MAE,RMSE,R²,Best Params
0,2014,Linear Regression,1.230866,1.614104,0.895138,{}
1,2014,Ridge Regression,0.945825,1.263190,0.935777,{'model__alpha': 5.994842503189409}
2,2014,Random Forest,0.923618,1.237043,0.938408,"{'model__max_depth': 5, 'model__max_features':..."
3,2014,XGBoost,0.959731,1.290856,0.932933,"{'model__colsample_bytree': 0.9, 'model__learn..."
4,2015,Linear Regression,1.358254,1.700766,0.908036,{}
5,2015,Ridge Regression,1.653868,1.931703,0.881366,{'model__alpha': 21.544346900318846}
6,2015,Random Forest,3.583185,5.037881,0.193093,"{'model__max_depth': 5, 'model__max_features':..."
7,2015,XGBoost,3.699582,5.218270,0.134274,"{'model__colsample_bytree': 0.9, 'model__learn..."
8,2016,Linear Regression,1.267696,1.701329,0.937492,{}
9,2016,Ridge Regression,1.113199,1.475792,0.952966,{'model__alpha': 1.6681005372000592}


In [28]:
naive_raw_results = []

for w in outer_windows:

    test = df.iloc[
        w["test_indices"]
    ]

    y_true = test["Target_Next_Close"]

    # Today's close predicts tomorrow's close
    y_pred = test["gold close"].values

    metrics = regression_metrics(
        y_true,
        y_pred
    )

    naive_raw_results.append({
        "Test Year": w["test_year"],
        "Model": "Naive - Today's Close",
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R²": metrics["R²"],
        "Best Params": {}
    })

naive_raw_df = pd.DataFrame(
    naive_raw_results
)

raw_results_df = pd.concat(
    [
        raw_results_df,
        naive_raw_df
    ],
    ignore_index=True
)

In [29]:
raw_summary = (
    raw_results_df
    .groupby("Model")
    [["MAE", "RMSE", "R²"]]
    .mean()
    .sort_values("RMSE")
)

raw_summary.round(4)

,MAE,RMSE,R²
Model,,,
Naive - Today's Close,0.9798,1.3100,0.9587
Ridge Regression,1.3206,1.6849,0.9294
Linear Regression,1.4051,1.8064,0.9116
Random Forest,6.1396,7.4816,0.1325
XGBoost,6.2295,7.5992,0.1211


## Phase 2:

In [30]:
def build_linear_log():

    preprocessor = make_scaled_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ])


def build_ridge_log():

    preprocessor = make_scaled_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            Ridge()
        )
    ])


def build_rf_log():

    preprocessor = make_tree_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])


def build_xgb_log():

    preprocessor = make_tree_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])

In [31]:
log_results = []

for w in outer_windows:

    train_idx = w["train_indices"]
    test_idx = w["test_indices"]

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    X_train = train[log_features]
    y_train = train[log_target]

    X_test = test[log_features]
    y_test = test[log_target]

    inner_cv = PurgedTimeSeriesCV(
        n_splits=3,
        test_size=150,
        purge_size=1,
        embargo_size=1
    )

    models = {

        "Linear Regression": (
            build_linear_log(),
            None
        ),

        "Ridge Regression": (
            build_ridge_log(),
            RIDGE_GRID
        ),

        "Random Forest": (
            build_rf_log(),
            RF_REG_GRID
        ),

        "XGBoost": (
            build_xgb_log(),
            XGB_REG_GRID
        )
    }

    for model_name, (pipeline, grid) in models.items():

        print(
            f"{w['test_year']} | "
            f"{model_name}"
        )

        if grid is None:

            pipeline.fit(
                X_train,
                y_train
            )

            best_model = pipeline
            best_params = {}

        else:

            search = tune_model(
                pipeline,
                grid,
                X_train,
                y_train,
                inner_cv
            )

            best_model = search.best_estimator_
            best_params = search.best_params_

        y_pred = best_model.predict(
            X_test
        )

        metrics = regression_metrics(
            y_test,
            y_pred
        )

        log_results.append({
            "Test Year": w["test_year"],
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R²": metrics["R²"],
            "Best Params": best_params
        })

log_results_df = pd.DataFrame(
    log_results
)

2014 | Linear Regression
2014 | Ridge Regression
2014 | Random Forest
2014 | XGBoost
2015 | Linear Regression
2015 | Ridge Regression
2015 | Random Forest
2015 | XGBoost
2016 | Linear Regression
2016 | Ridge Regression
2016 | Random Forest
2016 | XGBoost
2017 | Linear Regression
2017 | Ridge Regression
2017 | Random Forest
2017 | XGBoost
2018 | Linear Regression
2018 | Ridge Regression
2018 | Random Forest
2018 | XGBoost
2019 | Linear Regression
2019 | Ridge Regression
2019 | Random Forest
2019 | XGBoost
2020 | Linear Regression
2020 | Ridge Regression
2020 | Random Forest
2020 | XGBoost
2021 | Linear Regression
2021 | Ridge Regression
2021 | Random Forest
2021 | XGBoost
2022 | Linear Regression
2022 | Ridge Regression
2022 | Random Forest
2022 | XGBoost
2023 | Linear Regression
2023 | Ridge Regression
2023 | Random Forest
2023 | XGBoost
2024 | Linear Regression
2024 | Ridge Regression
2024 | Random Forest
2024 | XGBoost


In [34]:
log_results_df

,Test Year,Model,MAE,RMSE,R²,Best Params
0,2014,Linear Regression,0.006804,0.009426,-0.021217,{}
1,2014,Ridge Regression,0.006654,0.009319,0.001837,{'model__alpha': 1000.0}
2,2014,Random Forest,0.006640,0.009332,-0.001019,"{'model__max_depth': 3, 'model__max_features':..."
3,2014,XGBoost,0.006631,0.009351,-0.005026,"{'model__colsample_bytree': 0.9, 'model__learn..."
4,2015,Linear Regression,0.007157,0.009248,-0.080847,{}
5,2015,Ridge Regression,0.006815,0.008937,-0.009437,{'model__alpha': 1000.0}
6,2015,Random Forest,0.006951,0.009065,-0.038668,"{'model__max_depth': 5, 'model__max_features':..."
7,2015,XGBoost,0.007095,0.009199,-0.069506,"{'model__colsample_bytree': 0.7, 'model__learn..."
8,2016,Linear Regression,0.008049,0.010729,-0.122932,{}
9,2016,Ridge Regression,0.007500,0.010185,-0.012002,{'model__alpha': 1000.0}


In [32]:
zero_return_results = []

for w in outer_windows:

    test = df.iloc[
        w["test_indices"]
    ]

    y_true = test["Target_Log_Return"]

    # Predict zero return
    y_pred = np.zeros(
        len(test)
    )

    metrics = regression_metrics(
        y_true,
        y_pred
    )

    zero_return_results.append({
        "Test Year": w["test_year"],
        "Model": "Naive - Zero Return",
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R²": metrics["R²"],
        "Best Params": {}
    })

zero_return_df = pd.DataFrame(
    zero_return_results
)

log_results_df = pd.concat(
    [
        log_results_df,
        zero_return_df
    ],
    ignore_index=True
)

In [33]:
log_summary = (
    log_results_df
    .groupby("Model")
    [["MAE", "RMSE", "R²"]]
    .mean()
    .sort_values("RMSE")
)

log_summary.round(6)

,MAE,RMSE,R²
Model,,,
Naive - Zero Return,0.006540,0.008739,-0.004575
XGBoost,0.006580,0.008798,-0.019683
Ridge Regression,0.006596,0.008807,-0.019783
Random Forest,0.006589,0.008809,-0.024235
Linear Regression,0.006822,0.009074,-0.077871


In [35]:
# Example:
# predicted return = r_hat

# predicted next close:
#
# P_hat(t+1) = P(t) * exp(r_hat)

def return_to_price(
    current_price,
    predicted_return
):

    return (
        current_price
        * np.exp(predicted_return)
    )

In [36]:
predicted_price = return_to_price(
    test["gold close"].values,
    y_pred
)

In [37]:
predicted_price[:5], test["Target_Next_Close"].values[:5]

(array([190.72, 189.13, 189.32, 189.35, 187.87]),
 array([189.13, 189.32, 189.35, 187.87, 187.93]))

## Phase 3:

In [38]:
def build_logistic_classifier():

    preprocessor = make_scaled_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    model = LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        max_iter=5000,
        random_state=SEED
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])

In [39]:
LOGISTIC_GRID = {

    "model__C": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],

    "model__l1_ratio": [
        0.1,
        0.5,
        0.9
    ]
}

In [40]:
def build_rf_classifier():

    preprocessor = make_tree_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])

In [41]:
RF_CLF_GRID = {

    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        2,
        3,
        5,
        8
    ],

    "model__min_samples_leaf": [
        5,
        10,
        20
    ],

    "model__max_features": [
        "sqrt",
        0.5
    ]
}

In [42]:
def build_xgb_classifier():

    preprocessor = make_tree_preprocessor(
        log_numeric_features,
        log_categorical_features
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=SEED,
                n_jobs=-1
            )
        )
    ])

In [43]:
XGB_CLF_GRID = {

    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        2,
        3,
        5
    ],

    "model__learning_rate": [
        0.01,
        0.05
    ],

    "model__subsample": [
        0.7,
        0.9
    ],

    "model__colsample_bytree": [
        0.7,
        0.9
    ],

    "model__reg_lambda": [
        1,
        10
    ]
}

In [44]:
def classification_metrics(
    y_true,
    y_pred,
    y_prob
):

    return {

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Balanced Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob
        )
    }

In [45]:
def tune_classifier(
    pipeline,
    param_grid,
    X_train,
    y_train,
    cv
):

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    return search

In [46]:
classification_results = []

for w in outer_windows:

    train_idx = w["train_indices"]
    test_idx = w["test_indices"]

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    X_train = train[log_features]
    y_train = train[classification_target]

    X_test = test[log_features]
    y_test = test[classification_target]

    inner_cv = PurgedTimeSeriesCV(
        n_splits=3,
        test_size=150,
        purge_size=1,
        embargo_size=1
    )

    models = {

        "Logistic Regression": (
            build_logistic_classifier(),
            LOGISTIC_GRID
        ),

        "Random Forest": (
            build_rf_classifier(),
            RF_CLF_GRID
        ),

        "XGBoost": (
            build_xgb_classifier(),
            XGB_CLF_GRID
        )
    }

    for model_name, (pipeline, grid) in models.items():

        print(
            f"{w['test_year']} | "
            f"{model_name}"
        )

        search = tune_classifier(
            pipeline,
            grid,
            X_train,
            y_train,
            inner_cv
        )

        best_model = search.best_estimator_

        # Predictions
        y_pred = best_model.predict(
            X_test
        )

        # Probability of class 1
        y_prob = best_model.predict_proba(
            X_test
        )[:, 1]

        metrics = classification_metrics(
            y_test,
            y_pred,
            y_prob
        )

        classification_results.append({

            "Test Year": w["test_year"],

            "Model": model_name,

            "Accuracy": metrics["Accuracy"],

            "Balanced Accuracy":
                metrics["Balanced Accuracy"],

            "Precision":
                metrics["Precision"],

            "Recall":
                metrics["Recall"],

            "F1":
                metrics["F1"],

            "ROC-AUC":
                metrics["ROC-AUC"],

            "Best Params":
                search.best_params_
        })

classification_results_df = pd.DataFrame(
    classification_results
)

2014 | Logistic Regression
2014 | Random Forest
2014 | XGBoost
2015 | Logistic Regression
2015 | Random Forest
2015 | XGBoost
2016 | Logistic Regression
2016 | Random Forest
2016 | XGBoost
2017 | Logistic Regression
2017 | Random Forest
2017 | XGBoost
2018 | Logistic Regression
2018 | Random Forest
2018 | XGBoost
2019 | Logistic Regression
2019 | Random Forest
2019 | XGBoost
2020 | Logistic Regression
2020 | Random Forest
2020 | XGBoost
2021 | Logistic Regression
2021 | Random Forest
2021 | XGBoost
2022 | Logistic Regression
2022 | Random Forest
2022 | XGBoost
2023 | Logistic Regression
2023 | Random Forest
2023 | XGBoost
2024 | Logistic Regression
2024 | Random Forest
2024 | XGBoost


In [47]:
classification_results_df

,Test Year,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Best Params
0,2014,Logistic Regression,0.507937,0.515259,0.494565,0.745902,0.594771,0.499811,"{'model__C': 1, 'model__l1_ratio': 0.1}"
1,2014,Random Forest,0.492063,0.498613,0.483146,0.704918,0.573333,0.458953,"{'model__max_depth': 8, 'model__max_features':..."
2,2014,XGBoost,0.488095,0.491488,0.477124,0.598361,0.530909,0.464250,"{'model__colsample_bytree': 0.9, 'model__learn..."
3,2015,Logistic Regression,0.492063,0.518049,0.460317,0.769912,0.576159,0.536194,"{'model__C': 0.01, 'model__l1_ratio': 0.1}"
4,2015,Random Forest,0.460317,0.490100,0.442211,0.778761,0.564103,0.492265,"{'model__max_depth': 2, 'model__max_features':..."
5,2015,XGBoost,0.452381,0.475457,0.431694,0.699115,0.533784,0.478958,"{'model__colsample_bytree': 0.7, 'model__learn..."
6,2016,Logistic Regression,0.519841,0.523690,0.553846,0.281250,0.373057,0.517956,"{'model__C': 0.01, 'model__l1_ratio': 0.1}"
7,2016,Random Forest,0.523810,0.526966,0.552632,0.328125,0.411765,0.504032,"{'model__max_depth': 5, 'model__max_features':..."
8,2016,XGBoost,0.515873,0.516507,0.525862,0.476562,0.500000,0.514365,"{'model__colsample_bytree': 0.7, 'model__learn..."
9,2017,Logistic Regression,0.430279,0.497734,0.500000,0.013986,0.027211,0.581456,"{'model__C': 0.1, 'model__l1_ratio': 0.9}"


In [48]:
majority_results = []

for w in outer_windows:

    train = df.iloc[
        w["train_indices"]
    ]

    test = df.iloc[
        w["test_indices"]
    ]

    y_train = train[
        classification_target
    ]

    y_test = test[
        classification_target
    ]

    # Determine majority class from training only
    majority_class = (
        y_train
        .value_counts()
        .idxmax()
    )

    y_pred = np.full(
        len(y_test),
        majority_class
    )

    # Probability approximation
    class_probability = (
        y_train
        .mean()
    )

    y_prob = np.full(
        len(y_test),
        class_probability
    )

    metrics = classification_metrics(
        y_test,
        y_pred,
        y_prob
    )

    majority_results.append({

        "Test Year": w["test_year"],

        "Model": "Majority-Class Baseline",

        "Accuracy": metrics["Accuracy"],

        "Balanced Accuracy":
            metrics["Balanced Accuracy"],

        "Precision":
            metrics["Precision"],

        "Recall":
            metrics["Recall"],

        "F1":
            metrics["F1"],

        "ROC-AUC":
            metrics["ROC-AUC"],

        "Best Params": {}
    })

majority_df = pd.DataFrame(
    majority_results
)

classification_results_df = pd.concat(
    [
        classification_results_df,
        majority_df
    ],
    ignore_index=True
)

In [49]:
classification_summary = (
    classification_results_df
    .groupby("Model")
    [
        [
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC"
        ]
    ]
    .mean()
    .sort_values(
        "ROC-AUC",
        ascending=False
    )
)

classification_summary.round(4)

,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,,
Logistic Regression,0.5046,0.5046,0.5199,0.6832,0.5428,0.5105
XGBoost,0.4946,0.4983,0.5215,0.5440,0.5168,0.5077
Random Forest,0.5010,0.5039,0.5174,0.5887,0.5247,0.5027
Majority-Class Baseline,0.5060,0.5000,0.4222,0.8182,0.5562,0.5000


# Final Results:

In [ ]:
print("==========================================")
print("PHASE 1: RAW PRICE REGRESSION")
print("==========================================")

display(
    raw_summary.round(4)
)


print("==========================================")
print("PHASE 2: LOG-RETURN REGRESSION")
print("==========================================")

display(
    log_summary.round(6)
)


print("==========================================")
print("PHASE 3: DIRECTIONAL CLASSIFICATION")
print("==========================================")

display(
    classification_summary.round(4)
)

PHASE 1: RAW PRICE REGRESSION


,MAE,RMSE,R²
Model,,,
Naive - Today's Close,0.9798,1.3100,0.9587
Ridge Regression,1.3206,1.6849,0.9294
Linear Regression,1.4051,1.8064,0.9116
Random Forest,6.1396,7.4816,0.1325
XGBoost,6.2295,7.5992,0.1211


PHASE 2: LOG-RETURN REGRESSION


,MAE,RMSE,R²
Model,,,
Naive - Zero Return,0.006540,0.008739,-0.004575
XGBoost,0.006580,0.008798,-0.019683
Ridge Regression,0.006596,0.008807,-0.019783
Random Forest,0.006589,0.008809,-0.024235
Linear Regression,0.006822,0.009074,-0.077871


PHASE 3: DIRECTIONAL CLASSIFICATION


,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,,
Logistic Regression,0.5046,0.5046,0.5199,0.6832,0.5428,0.5105
XGBoost,0.4946,0.4983,0.5215,0.5440,0.5168,0.5077
Random Forest,0.5010,0.5039,0.5174,0.5887,0.5247,0.5027
Majority-Class Baseline,0.5060,0.5000,0.4222,0.8182,0.5562,0.5000
